# Recorded Gender Classification — Academic Demonstration
**GitHub repository:** https://github.com/St-virus1210/travel-analytics-capstone

This notebook fits Logistic Regression using age and company to predict the dataset's recorded gender label. This is sensitive-attribute prediction and must not be used for consequential decisions. Place users.csv in data/.

In [ ]:
from pathlib import Path
import json, joblib, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

users=pd.read_csv('data/users.csv').dropna(subset=['gender']).copy(); users['gender']=users['gender'].astype(str)
X=users[['age','company']].copy(); X['age']=pd.to_numeric(X['age'],errors='coerce'); X['company']=X['company'].fillna('unknown').astype(str); y=users['gender']
stratify=y if y.value_counts().min()>=2 else None
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=stratify)
prep=ColumnTransformer([('company',OneHotEncoder(handle_unknown='ignore'),['company']),('age',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),['age'])])
model=Pipeline([('preprocess',prep),('model',LogisticRegression(max_iter=1500,class_weight='balanced'))]); model.fit(X_train,y_train); pred=model.predict(X_test)
metrics={'accuracy':accuracy_score(y_test,pred),'balanced_accuracy':balanced_accuracy_score(y_test,pred),'macro_f1':f1_score(y_test,pred,average='macro',zero_division=0)}; display(pd.DataFrame([metrics])); print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,xticks_rotation=45); plt.tight_layout(); plt.show()
Path('artifacts').mkdir(exist_ok=True); joblib.dump(model,'artifacts/gender_classifier.joblib'); Path('artifacts/gender_metrics_notebook.json').write_text(json.dumps({'metrics':{k:float(v) for k,v in metrics.items()},'classification_report':classification_report(y_test,pred,zero_division=0)},indent=2)); print('Saved model and metrics.')

Interpret metrics against a majority-class baseline. The model may reflect dataset bias and weak signal; do not use it to infer identity or make decisions about people.